# 02 — Dados de março/2025
Reprodução: *Uma Proposta de Algoritmo para a Detecção de Mixers na Ethereum* (Leale & Sendin, SBSeg 2025).

| Bloco | O que faz |
|---|---|
| **A** | Baixa o dump do Blockchair de **15/03/2025** e converte para Parquet |
| **B** | Sorteia **150 candidatos a negativos** (não-mixer) para inspeção no Etherscan |
| **C** | Coleta os **positivos** (Tornado Cash 10 ETH, março/2025) pela API do Blockchair |

> Todas as células salvam no Drive e podem ser reexecutadas: se a sessão cair, rode a **Célula 1** e depois a célula onde parou.

## Célula 1 — Configuração

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd, numpy as np, requests, json, time, gc

BASE = Path('/content/drive/MyDrive/mixers-ethereum')
RAW, INTERIM, PROC, RES = [BASE/p for p in ['data/raw','data/interim','data/processed','results']]
for p in [RAW, INTERIM, PROC, RES, BASE/'docs', BASE/'notebooks']:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
TORNADO = {
    "0x12d66f87a04a9e220743712ce6d9bb1b5616b8fc": "0.1 ETH",
    "0x47ce0c6ed5b0ce3d3a51fdb1c52dc66a7c3c2936": "1 ETH",
    "0x910cbd523d972eb0a6f4cae4618ad62622b39dbf": "10 ETH",
}
print("Drive montado. Pasta base:", BASE)

Mounted at /content/drive
Drive montado. Pasta base: /content/drive/MyDrive/mixers-ethereum


---
## Bloco A — Dump de 15/03/2025

**Célula 2 — Tamanho do arquivo.** Divida o valor (bytes) por 10.000 para estimar o tempo em segundos a 10 kB/s. Anote em `docs/coleta.md`.

In [2]:
URL_2025 = "https://gz.blockchair.com/ethereum/transactions/blockchair_ethereum_transactions_20250315.tsv.gz"
!curl -sI "{URL_2025}" | grep -i content-length

Content-Length: 293598528


**Célula 3 — Download direto para o Drive.** O `-c` retoma de onde parou se a sessão cair. Não rode outro download em paralelo (o Blockchair só permite um por vez).

In [3]:
!wget -c --tries=0 --retry-connrefused -P "{RAW}" "{URL_2025}"

--2026-09-30 21:07:39--  https://gz.blockchair.com/ethereum/transactions/blockchair_ethereum_transactions_20250315.tsv.gz
Resolving gz.blockchair.com (gz.blockchair.com)... 217.115.120.73
Connecting to gz.blockchair.com (gz.blockchair.com)|217.115.120.73|:443... connected.
HTTP request sent, awaiting response... 416 Requested Range Not Satisfiable

    The file is already fully retrieved; nothing to do.



**Célula 3b — Acompanhar o progresso** (rode quando quiser).

In [4]:
!ls -lh "{RAW}"

total 511M
-rw------- 1 root root 229M Oct 29  2020 blockchair_ethereum_transactions_20201029.tsv.gz
-rw------- 1 root root 280M Mar 15  2025 blockchair_ethereum_transactions_20250315.tsv.gz
-rw------- 1 root root 417K Sep 29 18:29 tornado10_calls_2025-03-01.csv
-rw------- 1 root root   15 Sep 29 18:29 tornado10_calls_2025-03-01.estado.json
-rw------- 1 root root 220K Sep 29 18:29 tornado10_calls_2025-03-08.csv
-rw------- 1 root root   15 Sep 29 18:29 tornado10_calls_2025-03-08.estado.json
-rw------- 1 root root  64K Sep 29 18:30 tornado10_calls_2025-03-15.csv
-rw------- 1 root root   14 Sep 29 18:30 tornado10_calls_2025-03-15.estado.json
-rw------- 1 root root  75K Sep 29 18:30 tornado10_calls_2025-03-22.csv
-rw------- 1 root root   14 Sep 29 18:30 tornado10_calls_2025-03-22.estado.json
-rw------- 1 root root  46K Sep 29 18:30 tornado10_calls_2025-03-29.csv
-rw------- 1 root root   14 Sep 29 18:30 tornado10_calls_2025-03-29.estado.json
-rw------- 1 root root  48K Sep 29 18:29 tornado1

**Célula 4 — Integridade e colunas.** Confira se os nomes usados na Célula 5 (`COLS`) aparecem na lista.

In [5]:
f25 = RAW/"blockchair_ethereum_transactions_20250315.tsv.gz"
!gzip -t "{f25}" && echo "OK: arquivo íntegro"
!sha256sum "{f25}" | tee -a "{BASE}/docs/coleta.md"
!zcat "{f25}" | head -1 | tr '\t' '\n' | nl

OK: arquivo íntegro
01e6037103e2cad295a49b1c76eddf36676d68bb475f1f5c212502a8e7725a61  /content/drive/MyDrive/mixers-ethereum/data/raw/blockchair_ethereum_transactions_20250315.tsv.gz
     1	block_id
     2	index
     3	hash
     4	time
     5	failed
     6	type
     7	sender
     8	recipient
     9	call_count
    10	value
    11	value_usd
    12	internal_value
    13	internal_value_usd
    14	fee
    15	fee_usd
    16	gas_used
    17	gas_limit
    18	gas_price
    19	input_hex
    20	nonce
    21	v
    22	r
    23	s


**Célula 5 — Leitura em pedaços → Parquet.** Lê 250 mil linhas por vez para não estourar a RAM e converte wei → ETH.

In [6]:
COLS = ["hash","time","failed","sender","recipient","call_count","value","internal_value","fee"]
DT = {"value": str, "internal_value": str, "fee": str, "sender": str, "recipient": str}

partes = []
for chunk in pd.read_csv(f25, sep="\t", usecols=COLS, dtype=DT, chunksize=250_000):
    for col in ["value", "internal_value", "fee"]:
        chunk[col + "_eth"] = pd.to_numeric(chunk[col], errors="coerce") / 1e18   # wei -> ETH
    partes.append(chunk.drop(columns=["value", "internal_value", "fee"]))
    print(f"{sum(len(p) for p in partes):,} linhas lidas")

df25 = pd.concat(partes, ignore_index=True); del partes; gc.collect()
df25.to_parquet(INTERIM/"dia_20250315.parquet")

print("Transações:", len(df25))
print("Endereços distintos:", pd.concat([df25.sender, df25.recipient]).nunique())
print(df25[df25.recipient.isin(TORNADO)].recipient.map(TORNADO).value_counts())

250,000 linhas lidas
500,000 linhas lidas
750,000 linhas lidas
1,000,000 linhas lidas
1,009,770 linhas lidas
Transações: 1009770
Endereços distintos: 392290
Series([], Name: count, dtype: int64)


> Se a sessão cair depois desta célula, recarregue sem reler o `.tsv.gz`:

In [7]:
df25 = pd.read_parquet(INTERIM/"dia_20250315.parquet")

---
## Bloco B — Candidatos a negativos

**Célula 6 — Sorteio com semente fixa.** Critério: endereços que **receberam ≥ 10 transações** no dia, excluindo o Tornado. Registre esse critério em `docs/decisoes.md` (o artigo não o especifica).

Depois: abra o CSV no Google Sheets (**Arquivo → Importar**), troque *Membro 1/2/3* pelos nomes e inspecionem cada endereço no Etherscan. Aceitem os **100 primeiros** aprovados, na ordem do sorteio.

In [8]:
cont = df25.recipient.value_counts()
elegiveis = cont[(cont >= 10) & (~cont.index.isin(list(TORNADO)))]
print("Endereços elegíveis (>=10 transações recebidas):", len(elegiveis))

cand = elegiveis.index.to_series().sample(n=150, random_state=SEED).reset_index(drop=True)
planilha = pd.DataFrame({
    "ordem": range(1, 151),
    "endereco": cand,
    "etherscan": "https://etherscan.io/address/" + cand,
    "n_tx_no_dia": cand.map(cont).values,
    "responsavel": ["Bruno"]*50 + ["Gustavo"]*50 + ["Marília"]*50,
    "tipo": "", "rotulo_etherscan": "", "decisao": "", "motivo": "",
})
planilha.to_csv(INTERIM/"candidatos_negativos.csv", index=False)
planilha.head()

Endereços elegíveis (>=10 transações recebidas): 4704


,ordem,endereco,etherscan,n_tx_no_dia,responsavel,tipo,rotulo_etherscan,decisao,motivo
0,1,0x5ccdcd02c2a131243227e6db746c2315596d2cb0,https://etherscan.io/address/0x5ccdcd02c2a1312...,10,Bruno,,,,
1,2,0xd4517dbeef443cf3d1d760abdd4268f12940c6f8,https://etherscan.io/address/0xd4517dbeef443cf...,14,Bruno,,,,
2,3,0x898bf05dfec6a2cfbfda2877ff759d62beed7fe7,https://etherscan.io/address/0x898bf05dfec6a2c...,51,Bruno,,,,
3,4,0x54692b70dd7548748d31635c1b1da4c5bd971094,https://etherscan.io/address/0x54692b70dd75487...,12,Bruno,,,,
4,5,0xec2a8bef7afdebe0f807682be5cea98d95360be2,https://etherscan.io/address/0xec2a8bef7afdebe...,68,Bruno,,,,


---
## Bloco C — Positivos: Tornado Cash 10 ETH (API Blockchair)

Limites do plano gratuito: **1440 pontos/dia** e **30 requisições/minuto**, contados por IP. O IP do Colab é compartilhado; se aparecer código **429 / 430 / 434**, rode as Células 7–10 no computador de um membro e suba os CSVs para `data/raw/`.

**Célula 7 — Teste (1 requisição).** Confira `code` = 200, `total_rows` e `request_cost`. Se der erro de sintaxe na data, teste `time(2025-03-01)`.

In [9]:
API = "https://api.blockchair.com/ethereum"
T10 = "0x910cbd523d972eb0a6f4cae4618ad62622b39dbf"
r = requests.get(f"{API}/transactions",
                 params={"q": f"recipient({T10}),time(2025-03-01..2025-03-31)", "limit": 1},
                 timeout=60).json()
print(json.dumps(r.get("context", {}), indent=1)[:1000])

{
 "code": 200,
 "source": "A+T",
 "limit": 1,
 "offset": 0,
 "rows": 1,
 "pre_rows": 1,
 "total_rows": 121,
 "state": 26092883,
 "state_layer_2": 26092883,
 "market_price_usd": 2680.7,
 "cache": {
  "live": true,
  "duration": 120,
  "since": "2026-09-30 21:08:32",
  "until": "2026-09-30 21:10:32",
  "time": null
 },
 "api": {
  "version": "2.0.95-ie",
  "last_major_update": "2022-11-07 02:00:00",
  "next_major_update": "2023-11-12 02:00:00",
  "documentation": "https://blockchair.com/api/docs",
  "notice": "Try out our new API v.3: https://3xpl.com/data"
 },
 "servers": "API4,ETH4",
 "time": 0.054183006286621094,
 "render_time": 0.002721071243286133,
 "full_time": 0.05690407752990723,
 "request_cost": 7
}


**Célula 8 — Função de coleta com retomada.** Grava cada página no Drive e o último `offset` em `.estado.json`; ao rodar de novo (mesmo em outro dia), continua de onde parou.

In [10]:
def coletar(tabela, q, saida, limite=100, pausa=3):
    """Pagina a API e grava cada página no Drive; se cair, continua do último offset salvo."""
    estado = saida.with_name(saida.stem + ".estado.json")
    offset = json.loads(estado.read_text())["offset"] if estado.exists() else 0
    while True:
        r = requests.get(f"{API}/{tabela}", params={"q": q, "limit": limite, "offset": offset},
                         timeout=60).json()
        ctx = r.get("context", {})
        if ctx.get("code") != 200:
            print(f"Parou em offset {offset}: código {ctx.get('code')} - {ctx.get('error')}")
            return False
        dados = r["data"]
        if dados:
            pd.DataFrame(dados).to_csv(saida, mode="a", header=not saida.exists(), index=False)
        offset += len(dados)
        estado.write_text(json.dumps({"offset": offset}))
        print(f"{offset}/{ctx['total_rows']}  custo={ctx.get('request_cost')}")
        if len(dados) < limite:
            print("Concluído:", saida.name)
            return True
        time.sleep(pausa)

SEMANAS = ["2025-03-01..2025-03-07", "2025-03-08..2025-03-14", "2025-03-15..2025-03-21",
           "2025-03-22..2025-03-28", "2025-03-29..2025-03-31"]

**Célula 9 — Transações diretas ao contrato.**

In [11]:
for sem in SEMANAS:
    ok = coletar("transactions", f"recipient({T10}),time({sem})",
                 RAW/f"tornado10_tx_{sem[:10]}.csv")
    if not ok: break          # cota estourou: rode de novo depois (continua do ponto salvo)

31/31  custo=13.299999999999999
Concluído: tornado10_tx_2025-03-01.csv
90/90  custo=13.299999999999999
Concluído: tornado10_tx_2025-03-08.csv
0/0  custo=13.299999999999999
Concluído: tornado10_tx_2025-03-15.csv
0/0  custo=13.299999999999999
Concluído: tornado10_tx_2025-03-22.csv
0/0  custo=13.299999999999999
Concluído: tornado10_tx_2025-03-29.csv


**Célula 10 — Chamadas internas (verificação do Tornado Router).** Custa o dobro por requisição; de preferência rode em outro dia.

In [12]:
for sem in SEMANAS:
    ok = coletar("calls", f"recipient({T10}),time({sem})",
                 RAW/f"tornado10_calls_{sem[:10]}.csv")
    if not ok: break

492/492  custo=22.799999999999997
Concluído: tornado10_calls_2025-03-01.csv
285/285  custo=22.799999999999997
Concluído: tornado10_calls_2025-03-08.csv
78/78  custo=22.799999999999997
Concluído: tornado10_calls_2025-03-15.csv
93/93  custo=22.799999999999997
Concluído: tornado10_calls_2025-03-22.csv
56/56  custo=22.799999999999997
Concluído: tornado10_calls_2025-03-29.csv


**Célula 11 — Comparação.** Se houver muitas transações só via Router, o grupo decide se inclui (registrar em `decisoes.md`).

In [ ]:
tx    = pd.concat(pd.read_csv(p) for p in sorted(RAW.glob("tornado10_tx_*.csv")))
calls = pd.concat(pd.read_csv(p) for p in sorted(RAW.glob("tornado10_calls_*.csv")))

so_router = set(calls.transaction_hash) - set(tx.hash)
print(f"{len(tx)} transações diretas ao contrato")
print(f"{len(so_router)} transações que só chegam ao contrato como chamada interna (via Router)")
pd.Series(sorted(so_router), name="hash").to_csv(INTERIM/"tornado10_hashes_via_router.csv", index=False)
tx.to_parquet(INTERIM/"tornado10_2025_03.parquet")